In [1]:
import pandas as pd
import torch
import numpy as np
from torch.utils.data import Dataset, DataLoader

# Load UCI promoter dataset directly
url = "https://archive.ics.uci.edu/ml/machine-learning-databases/molecular-biology/promoter-gene-sequences/promoters.data"

df = pd.read_csv(url, header=None, names=['label', 'id', 'sequence'])

In [2]:
def clean_sequence(seq):
    seq = seq.upper()
    seq = seq.strip()

    return seq
df_cleaned = df.copy()
df_cleaned['label'] = df_cleaned['label'].map({'-': 0, '+': 1})
df_cleaned['sequence'] = df_cleaned['sequence'].apply(clean_sequence)

In [3]:
import sys
sys.path.append('../src')
from UpdatedSequenceEncoder import SequenceEncoder
from sklearn.model_selection import train_test_split

class PromoterDataset(Dataset):
    def __init__(self, sequences, labels, encoder):
        self.sequences = sequences
        self.labels = labels
        self.encoder = encoder
        self.encoded_sequences = [self.encoder.encode(seq).T for seq in self.sequences]
        self.encoded_labels = torch.tensor(self.labels, dtype=torch.float32)
    
    def __len__(self):
        return len(self.encoded_sequences)
    
    def __getitem__(self, idx):
        return self.encoded_sequences[idx], self.encoded_labels[idx]
    
# Prepare data
sequences = df_cleaned['sequence'].tolist()
labels = df_cleaned['label']
encoder = SequenceEncoder()
dataset = PromoterDataset(sequences, labels, encoder)
X_train, X_test, y_train, y_test = train_test_split(dataset.encoded_sequences, dataset.encoded_labels, test_size=0.2, random_state=42)

X_train[10].shape, y_train[10].shape

(torch.Size([4, 57]), torch.Size([]))

In [10]:
import torch
import sys
sys.path.append('../src')
from LSTMClassifier import LSTMClassifier as Model

model = Model(input_size=4, hidden_size=57)
criterion = torch.nn.BCELoss()
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)
model.train()
#Track loss across epochs
for epoch in range(50):
    epoch_loss = 0
    for idx, sequence in enumerate(X_train):
        output = model(sequence.T)
        loss = criterion(output.squeeze(-1), y_train[idx].unsqueeze(-1))
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        epoch_loss += loss.item()
    if (epoch +1) % 10 == 0:  # Print loss every 10 epochs
        print(f"Epoch {epoch + 1}, Loss: {epoch_loss / len(X_train)}")

#Evaluate on test set
model.eval()
with torch.no_grad():
    correct = 0
    total = 0
    for idx, sequence in enumerate(X_test):
        output = model(sequence.T)
        predicted = (output.squeeze(-1) > 0.5).float()
        correct += (predicted == y_test[idx].unsqueeze(-1)).sum().item()
        total += 1
    print(f"Test Accuracy: {correct / total:.4f}")


Epoch 10, Loss: 0.6974297988982427
Epoch 20, Loss: 0.6145895983846414
Epoch 30, Loss: 0.658169720854078
Epoch 40, Loss: 0.4991442412137985
Epoch 50, Loss: 0.3468166594837038
Test Accuracy: 0.5909


In [ ]:
import torch
import sys
sys.path.append('../src')
from LSTMTrainer import Model

model = Model(input_size=4, hidden_size=57)
criterion = torch.nn.BCELoss()
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)
model.train()
#Track loss across epochs
for epoch in range(50):
    epoch_loss = 0
    for idx, sequence in enumerate(X_train):
        output = model(sequence.T)
        loss = criterion(output.squeeze(-1), y_train[idx].unsqueeze(-1))
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        epoch_loss += loss.item()
    if (epoch +1) % 10 == 0:  # Print loss every 10 epochs
        print(f"Epoch {epoch + 1}, Loss: {epoch_loss / len(X_train)}")

#Evaluate on test set
model.eval()
with torch.no_grad():
    correct = 0
    total = 0
    for idx, sequence in enumerate(X_test):
        output = model(sequence.T)
        predicted = (output.squeeze(-1) > 0.5).float()
        correct += (predicted == y_test[idx].unsqueeze(-1)).sum().item()
        total += 1
    print(f"Test Accuracy: {correct / total:.4f}")
